# Laboratory: Sentence Boundary Disambiguation with spaCy and NLTK

## Overview and Theoretical Foundations
Sentence Boundary Disambiguation (SBD), also known as sentence segmentation or sentence boundary detection, is the task of dividing a continuous stream of text into discrete sentences. SBD is a critical preprocessing prerequisite for downstream NLP applications, including syntactic parsing, discourse analysis, machine translation, text summarization, and relation extraction.

### The Challenge of Sentence Boundary Disambiguation
Splitting sentences by simply matching period (`.`), question mark (`?`), or exclamation mark (`!`) characters fails frequently in natural text because punctuation marks are polysemous:
- **Honorifics and Abbreviations:** Titles (`Mr.`, `Dr.`, `Prof.`), abbreviations (`e.g.`, `i.e.`, `etc.`), and geographical names (`U.S.`, `U.K.`) contain periods that do not denote sentence termination.
- **Numerical Quantities:** Decimal values (`3.26 dollars`, `10.30 a.m.`) introduce periods embedded inside numeric tokens.
- **Web Entities:** Domain names (`example.com`) and URLs contain unspaced punctuation.
- **Parentheses, Semicolons, and Quotes:** Quotation marks often wrap ending punctuation, obscuring boundary positions.

### Architectural Paradigms: Statistical Dependency Parsing vs. Unsupervised Punkt
- **spaCy Dependency-Driven Segmentation:** In standard spaCy pipelines (`en_core_web_sm`), sentence boundary detection is performed by the statistical transition-based dependency parser. A token is marked as a sentence head or start based on syntactic tree dependencies. spaCy also allows users to register custom rule-based components (`@Language.component`) that set `token.is_sent_start` before parser execution.
- **NLTK Punkt Algorithm:** NLTK implements the unsupervised Punkt algorithm (Kiss and Strunk, 2006). Punkt uses heuristic collocations, orthographic casing evidence, and abbreviation lexicons to distinguish abbreviation periods from sentence-final periods without requiring annotated treebanks.


---
## 1. Environment Setup and Linguistic Model Initialization

We configure our execution runtime with `spacy` and `nltk`, loading the small English pipeline `en_core_web_sm` with fallback downloading.


In [ ]:
%pip install spacy nltk


In [ ]:
import sys 

# Install spaCy small English pipeline
!{sys.executable} -m spacy download en_core_web_sm


In [ ]:
import spacy

# Load the small English model with fallback
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("spaCy version:", spacy.__version__)
print("Active pipeline:", nlp.pipe_names)


spaCy version: 3.8.18
Active pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


---
## 2. Sentence Segmentation in spaCy via `Doc.sents`

When a document is processed by spaCy, sentence boundaries are automatically calculated.
- Sentences are accessed via the `doc.sents` generator.
- Each generated item is a `Span` object representing the token sequence of that sentence.


In [ ]:
text = (
    "This is the first sentence. "
    "This is another sentence. "
    "This is the last sentence."
)
doc = nlp(text)

for i, sent in enumerate(doc.sents, start=1):
    print(f"Sentence {i}: {sent.text}")


Sentence 1: This is the first sentence.
Sentence 2: This is another sentence.
Sentence 3: This is the last sentence.


---
## 3. Sentence `Span` Objects and Token Offsets

Each sentence yielded by `doc.sents` is an instance of `spacy.tokens.Span`.
- `Span.start`: Starting token index within the parent `Doc`.
- `Span.end`: Non-inclusive ending token index within the parent `Doc`.
- Evaluating span properties enables direct token and character level indexing without re-parsing.


In [ ]:
sentences = list(doc.sents)

print("Total sentence count:", len(sentences))
print("Second sentence text:", sentences[1].text)
print("Second sentence type:", type(sentences[1]))
print(f"Token slice interval: [{sentences[1].start}, {sentences[1].end})")

print("
Individual tokens in parent document:")
for token in doc:
    print(f"Index {token.i:<2} | Text: {token.text}")


Error: unterminated string literal (detected at line 8) (<string>, line 8)


---
## 4. Detecting Sentence Boundaries via `token.is_sent_start`

Internally, spaCy represents sentence boundaries via the boolean flag `token.is_sent_start`:
- `True`: The token explicitly initiates a new sentence.
- `False`: The token is interior to an ongoing sentence.
- `None`: Sentence boundaries have not yet been evaluated by any pipeline component.


In [ ]:
text = "This is a sentence. That is another sentence. Here is a third"
doc = nlp(text)

for token in doc:
    print(f"{token.text:10} | is_sent_start={token.is_sent_start}")


This       | is_sent_start=True
is         | is_sent_start=False
a          | is_sent_start=False
sentence   | is_sent_start=False
.          | is_sent_start=False
That       | is_sent_start=True
is         | is_sent_start=False
another    | is_sent_start=False
sentence   | is_sent_start=False
.          | is_sent_start=False
Here       | is_sent_start=True
is         | is_sent_start=False
a          | is_sent_start=False
third      | is_sent_start=False


---
## 5. Ambiguity Challenges in Sentence Boundary Detection

Rule-based heuristics (such as splitting strictly on `.`) fail when encountering:
1. **Honorifics:** `"Mr. Smith"` should not be split at the abbreviation period.
2. **Numeric Decimals:** `"3.26 dollars"` embeds a decimal period inside a quantity.
3. **Web Domains and Spacing Anomalies:** `"example. com"` or malformed URLs.

spaCy's statistical model parses these syntactic structures correctly in most standard contexts.


In [ ]:
examples = [
    "Hello Mr. Smith, how are you today?",
    "The value is 3.26 dollars. The meeting is on Monday.",
    "Visit example. com for more information."
]

for text in examples:
    print("
Target Text:", text)
    for sent in nlp(text).sents:
        print("  - Sentence:", sent.text)


Error: unterminated string literal (detected at line 8) (<string>, line 8)


---
## 6. Extending spaCy with Custom Pipeline Components

In legal, medical, or corporate documentation, specialized punctuation (such as semicolons `;` or bullet sequences) may indicate distinct propositions that should be segmented as separate sentences.

We define a custom pipeline component using `@Language.component`:
- Inspects tokens for semicolons `;`.
- Explicitly flags the subsequent token with `is_sent_start = True`.
- Injects the component before the parser (`before="parser"`) so that subsequent syntactic processing honors the custom boundary.


In [ ]:
from spacy.language import Language

@Language.component("semicolon_sentence_boundary")
def semicolon_sentence_boundary(doc):
    for token in doc[:-1]:
        if token.text == ";":
            doc[token.i + 1].is_sent_start = True
    return doc

# Register component before the parser if not already present
if "semicolon_sentence_boundary" not in nlp.pipe_names:
    nlp.add_pipe("semicolon_sentence_boundary", before="parser")

print("Pipeline components:", nlp.pipe_names)

custom_text = '''Management is doing things right;
leadership is doing the right things.'''

custom_doc = nlp(custom_text)
print("
Segmented sentences with custom boundary:")
for sent in custom_doc.sents:
    print("  -", sent.text.strip())


Error: unterminated string literal (detected at line 20) (<string>, line 20)


---
## 7. Sentence Segmentation with NLTK and the Punkt Model

NLTK provides the `sent_tokenize()` function, which uses the unsupervised Punkt Sentence Boundary Detector.
- Punkt builds internal models of common abbreviations, sentence starters, and collocations without requiring a full syntactic dependency tree.
- Requires downloading the `punkt` and `punkt_tab` resources.


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

NLTK_RESOURCES = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab"
}

for name, path in NLTK_RESOURCES.items():
    try:
        nltk.data.find(path)
    except LookupError:
        print(f"Resource '{name}' not found. Downloading...")
        nltk.download(name, quiet=True)

example_text = (
    "Hello Mr. Smith, how are you today? "
    "The weather is great, and python is awesome. "
    "The sky is pinkish-blue. You shouldn't eat cardboard"
)

for i, sentence in enumerate(sent_tokenize(example_text), start=1):
    print(f"{i}. {sentence}")


Resource 'punkt' not found. Downloading...
[nltk_data] Error loading punkt: <urlopen error pathsec.urlopen: no
[nltk_data]     validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
Resource 'punkt_tab' not found. Downloading...
[nltk_data] Error loading punkt_tab: <urlopen error pathsec.urlopen:
[nltk_data]     no validated address for host
[nltk_data]     'raw.githubusercontent.com'; refusing to connect by
[nltk_data]     unvalidated hostname>
Error: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/home/youssef/.local/share/nltk_data'
    - '/home/youssef/nltk_data'
    - '/home/youssef/Projects/academic/nlp/0

---
## 8. Comparative Evaluation: spaCy vs. NLTK on Complex Input

We test both frameworks on a challenging paragraph containing:
- Abbreviation titles (`Dr.`)
- Embedded time periods (`10.30 a.m.`)
- Multi-sentence declarations and exclamations (`!`)


In [ ]:
text = (
    "Dr. Smith arrived at 10.30 a.m. "
    "He gave a short talk. Everyone enjoyed it!"
)

print("=== spaCy Segmentation ===")
for sent in nlp(text).sents:
    print(" -", sent.text)

print("
=== NLTK Segmentation ===")
for sent in sent_tokenize(text):
    print(" -", sent)


Error: unterminated string literal (detected at line 10) (<string>, line 10)
